In [10]:
import json
path = '/home/khemra/projects/crispy-sniffle/equity_msft/research_loop/supervisor_last_run_grok_2025_wf.json'
with open(path) as f:
    data = json.load(f)
text = data["last_grok_eval"].encode("latin-1").decode("utf-8")
print(text)

1. Structural Strengths
- Positive sample: total_return 0.186, CAGR 0.0216, Sharpe 0.53, profit_factor 1.90 over 2018-01-02 to 2025-12-31.
- Payoff asymmetry: win_rate 0.449 with PF 1.90 implies average winners larger than average losers.
- Risk tightly bound: max_drawdown 0.0482; equity_peak 124063 vs equity_final 118587.
- Sample size is usable: num_trades 196 (~25/year), so the result is not a no-trade artifact.

2. Structural Weaknesses
- Main defect is cash drag, not risk: buy_hold_return 4.627 vs total_return 0.186. A higher Sharpe from staying out is not an improvement.
- CAGR 2.16% does not participate in the rising tape that produced BH +463%.
- Peak-to-final fade: 124063 → 118587 while MDD stayed ~4.8%, consistent with clipped upside rather than a crash.
- Hit rate is not the edge (WR 44.9%); the book needs larger captured trends, not more small wins.

3. Regime Vulnerabilities
- No regime-split metrics were provided; any year-by-year claim would be invented.
- Aggregate impl

In [13]:
from IPython.display import Markdown, display

display(Markdown(data["last_grok_eval"].encode("latin-1").decode("utf-8")))

1. Structural Strengths
- Positive sample: total_return 0.186, CAGR 0.0216, Sharpe 0.53, profit_factor 1.90 over 2018-01-02 to 2025-12-31.
- Payoff asymmetry: win_rate 0.449 with PF 1.90 implies average winners larger than average losers.
- Risk tightly bound: max_drawdown 0.0482; equity_peak 124063 vs equity_final 118587.
- Sample size is usable: num_trades 196 (~25/year), so the result is not a no-trade artifact.

2. Structural Weaknesses
- Main defect is cash drag, not risk: buy_hold_return 4.627 vs total_return 0.186. A higher Sharpe from staying out is not an improvement.
- CAGR 2.16% does not participate in the rising tape that produced BH +463%.
- Peak-to-final fade: 124063 → 118587 while MDD stayed ~4.8%, consistent with clipped upside rather than a crash.
- Hit rate is not the edge (WR 44.9%); the book needs larger captured trends, not more small wins.

3. Regime Vulnerabilities
- No regime-split metrics were provided; any year-by-year claim would be invented.
- Aggregate implication only: over a window where MSFT BH compounded +463%, a long-only book at +18.6% / 4.8% MDD was mostly not aboard the dominant uptrend (including any post-shock recoveries in this span).
- Uncertainty: 2022-style weakness may be where the tight MDD was “earned,” but that is not measurable from these numbers.

4. Volatility-Specific Behavior
- No ATR/vol-bucket stats given; vol behavior is inferred only from outcome shape.
- 4.8% MDD vs BH +463% implies the strategy is absent from the large, often high-range advances that drove the tape.
- Existing vol/volume knobs (min_atr, extreme_setup_rel_vol, neutral_rel_volume_max) are the plausible gates; which one is binding is unknown.

5. Risk Model Evaluation
- Loss control is working (MDD 0.048, PF 1.90) and should stay controlled.
- Risk is over-tight relative to the opportunity: equity never compounded anywhere near BH.
- Do not treat the low MDD / modest Sharpe as success while total_return is an order of magnitude below buy_hold_return.

6. Edge Concentration
- Edge, as measured, sits in payoff asymmetry (PF 1.90), not frequency of wins.
- 196 trades means underperformance is not “too few trades”; it is insufficient hold/participation per trade versus a strong uptrend.
- No setup-level PnL was provided, so concentration across pullback vs extreme vs bearish vs neutral is unknown.

7. Failure Modes
- Primary: missing the rising tape (cash drag vs BH 4.627).
- Secondary: early de-risk (overbought / take-profit / hold-RSI) that would cap winners in a multi-year grind.
- Tertiary: entry filters (RSI / min_atr / rel vol / neutral band) blocking shallow-pullback and low-vol trend days.
- Observed giveback from equity_peak 124063 to 118587 without a large MDD spike.

8. Recommended Next Experiments
Aim at more time-in-market in the rising tape; keep MDD in check. Name knobs only; do not change code. Direction assumes standard naming (higher sell/TP thresholds = hold longer; higher buy_pullback_rsi = shallower entries). If a knob is inverted in the frozen code, reverse that one run.

- sell_overbought_rsi: step up (hold through higher RSI).
- take_profit_rsi: step up; take_profit_pnl_pct: step up (let winners extend).
- bullish_hold_rsi: step down (stay in bullish holds at lower RSI).
- buy_pullback_rsi: step up (accept shallower pullbacks).
- min_atr: step down (allow low-vol grind entries).
- neutral_rsi_low down / neutral_rsi_high up, and neutral_rel_volume_max up (widen quiet-tape participation).
- Leave bearish_entry_rsi, bearish_exit_rsi, sell_bearish_rsi, extreme_setup_rsi, extreme_setup_rel_vol unchanged until a participation-positive run is measured against buy_hold_return; those are more likely to add stand-aside behavior.

Success for the next run: total_return moves toward buy_hold_return with MDD still controlled. Do not score a win on Sharpe alone.

In [12]:
# Continue only if the 2026 holdout has total return above 0, more than one trade, and max drawdown at or under 12%.
# "total_return", "max_drawdown", "num_trades"


import json
import pandas as pd

path = '/home/khemra/projects/crispy-sniffle/equity_msft/research_loop/supervisor_last_run_grok_2025_wf.json'
with open(path) as f:
    data = json.load(f)

keys = ["total_return", "max_drawdown", "num_trades"]
holdout = data["holdout_metrics"]

df = pd.DataFrame(
    {"metric": keys, "value": [holdout[k] for k in keys]}
)
df


,metric,value
0,total_return,0.009775
1,max_drawdown,0.003237
2,num_trades,2.000000
